## minute price excess return

In [3]:
import pandas as pd
import numpy as np

# === Step 1: Load minute-level price data for GME and SPY ===
# gme = pd.read_csv('./GME-202011-202104-minute_price.csv', parse_dates=['datetime'])
spy = pd.read_csv('./SPY-202011-202104-minute_price.csv', parse_dates=['datetime'])
amc = pd.read_csv('./AMC-202011-202104-minute_price.csv', parse_dates=['datetime'])

# === Step 2: Rename price columns for clarity ===
# gme = gme.rename(columns={"last_trade_price": "gme_price"})
spy = spy.rename(columns={"last_trade_price": "spy_price"})
amc = amc.rename(columns={"last_trade_price": "amc_price"})

# === Step 3: Merge GME and SPY data on datetime ===
# df = pd.merge(gme[['datetime', 'gme_price']], spy[['datetime', 'spy_price']], on='datetime')
df = pd.merge(spy[['datetime', 'spy_price']], amc[['datetime', 'amc_price']], on='datetime')
df = df.sort_values(by='datetime').reset_index(drop=True)

# === Step 4: Compute log returns for GME and SPY ===
# df['gme_return'] = np.log(df['gme_price'] / df['gme_price'].shift(1))
df['spy_return'] = np.log(df['spy_price'] / df['spy_price'].shift(1))
df['amc_return'] = np.log(df['amc_price'] / df['amc_price'].shift(1))

# === Step 5: Calculate excess return (GME return minus SPY return) ===
df['excess_return'] = df['amc_return'] - df['spy_return']

# === Step 6: Set datetime as index for time-based rolling operations ===
df = df.set_index('datetime')

# === Step 7: Define multiple time windows for rolling cumulative excess return ===
windows = {
    '1min': '1min',
    '5min': '5min',
    '30min': '30min',
    '1h': '1h',
    '6h': '6h',
    '1d': '1d',
    '1w': '7d'  
}

# === Step 8: Calculate rolling sum of excess returns for each time window ===
for label, window in windows.items():
    df[f'excess_return_cum_{label}'] = df['excess_return'].rolling(window=window).sum()

# === Step 9: Reset index and save to CSV ===
df = df.reset_index()
df.to_csv("AMC-minute_price-excess-return.csv", index=False)

# === Step 10: Print a preview of selected columns for inspection ===
print(df[['datetime', 'excess_return', 'excess_return_cum_1min', 'excess_return_cum_5min', 
          'excess_return_cum_1h', 'excess_return_cum_1d']].dropna().head())


             datetime  excess_return  excess_return_cum_1min  \
1 2020-11-02 04:01:00      -0.022382               -0.022382   
2 2020-11-02 04:02:00       0.035353                0.035353   
3 2020-11-02 04:05:00      -0.001095               -0.001095   
4 2020-11-02 04:06:00       0.000061                0.000061   
5 2020-11-02 04:07:00       0.030209                0.030209   

   excess_return_cum_5min  excess_return_cum_1h  excess_return_cum_1d  
1               -0.022382             -0.022382             -0.022382  
2                0.012971              0.012971              0.012971  
3                0.011876              0.011876              0.011876  
4                0.034319              0.011937              0.011937  
5                0.029175              0.042147              0.042147  


In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import os

def plot_cumulative_excess_return(source_type='minute_price'):
    # === Validate input ===
    if source_type not in ['minute_price', 'midpoint']:
        raise ValueError("source_type must be 'minute_price' or 'midpoint'.")

    # === Define input file and output directory ===
    input_file = f"GME-{source_type}-excess-return.csv"
    output_dir = f"visualization/{source_type}"
    os.makedirs(output_dir, exist_ok=True)

    # === Load data and set datetime as index ===
    df = pd.read_csv(input_file, parse_dates=["datetime"])
    df = df.set_index("datetime")

    # === Columns to plot ===
    columns_to_plot = [
        'excess_return_cum_1min',
        'excess_return_cum_5min',
        'excess_return_cum_30min',
        'excess_return_cum_1h',
        'excess_return_cum_6h',
        'excess_return_cum_1d',
        'excess_return_cum_1w'
    ]

    # === Plot each cumulative excess return and save the figure ===
    for col in columns_to_plot:
        if col in df.columns:
            plt.figure(figsize=(12, 4))
            plt.plot(df.index, df[col], label=col)
            plt.title(f'Cumulative Excess Return - {col}')
            plt.xlabel('Datetime')
            plt.ylabel('Cumulative Excess Return')
            plt.legend()
            plt.grid(True)
            plt.tight_layout()
            plt.savefig(f"{output_dir}/{col}.png")
            plt.close()

    print(f"All plots have been saved to the folder: {output_dir}/")


In [ ]:
plot_cumulative_excess_return('minute_price')
plot_cumulative_excess_return('midpoint')


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5))
plt.plot(df['datetime'], df['excess_return'], color='blue', alpha=0.6)
plt.axhline(0, linestyle='--', color='gray')
plt.title('GME Minute-Level Excess Return Over Time')
plt.xlabel('Time')
plt.ylabel('Excess Return')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
max_row = df.loc[df['excess_return'].idxmax()]
min_row = df.loc[df['excess_return'].idxmin()]

print("Max excess return date:", max_row['datetime'])
print("value", max_row['excess_return'])

print("Min excess return date:", min_row['datetime'])
print("value", min_row['excess_return'])


In [ ]:
import seaborn as sns

plt.figure(figsize=(10, 5))
sns.histplot(df['excess_return'].dropna(), bins=100, kde=True, color='purple')
plt.title('Distribution of Excess Returns (GME - SPY)')
plt.xlabel('Excess Return')
plt.ylabel('Frequency')
plt.grid(True)
plt.tight_layout()
plt.show()
